In [ ]:
# 6회차 — smoke 클래스 추가 (v1, 2026-08-04)
# 이 배너가 보이면 최신본입니다. 위에서부터 순서대로 실행하세요. 전체 약 1시간 10분.
print('round6 notebook v1')
!nvidia-smi -L
!pip -q install ultralytics==8.3.* kagglehub

In [ ]:
# [1] 업로드 — kitchen-fire-poc.zip · assets_1~5.zip (한 번에 여러 개 선택 가능)
import zipfile, os, glob
from google.colab import files
up = files.upload()
os.makedirs('/content/work', exist_ok=True)
for n in up:
    zipfile.ZipFile(n).extractall('/content/work')
os.chdir('/content/work')
print(sorted(os.listdir('.')))
for d in ('bases','flamelib','smokelib','negsrc','eval_neg','eval_steam','weights'):
    p = f'assets/{d}'
    print(f'{d:10s}', len(glob.glob(p+'/*')) if os.path.isdir(p) else '없음')

In [ ]:
# [2] D-Fire 내려받기 (Kaggle 계정 필요)
import kagglehub, os
DFIRE = kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo')
print(DFIRE)
!ls "$DFIRE" 

In [ ]:
# [3] 평가셋 구성 — A-fire·C·학습용 배경, 그리고 A-smoke
!python scripts/dfire_eval_set.py --dfire "$DFIRE" --out eval
print('-'*60)
!python scripts/dfire_smoke_eval.py --dfire "$DFIRE" --out eval

In [ ]:
# [4] 합성 — fire·smoke 2클래스 학습셋 생성 (약 4분)
!python scripts/synthesize_smoke.py --assets assets --out ds6 \
    --dfire-bg-list eval/train_bg.txt --dfire-bg-count 600 --haze-prob 0.5
!cat ds6/data.yaml

In [ ]:
# [4-1] 합성 결과 눈으로 확인 — 라벨 박스를 그려서 12장
import cv2, glob, numpy as np, random, os
from google.colab.patches import cv2_imshow
fs = sorted(glob.glob('ds6/images/train/k*.jpg')); random.seed(0)
pick = lambda pat, n: random.sample([f for f in fs if pat in f], n)
sel = pick('_smoke.jpg', 4) + pick('fire_smoke', 4) + pick('_fire.jpg', 4)
COL = {0: (0, 255, 0), 1: (255, 180, 0)}
tiles = []
for f in sel:
    im = cv2.imread(f); H, W = im.shape[:2]
    for line in open(f.replace('/images/', '/labels/').replace('.jpg', '.txt')):
        c, x, y, bw, bh = line.split(); c = int(c); x, y, bw, bh = map(float, (x, y, bw, bh))
        cv2.rectangle(im, (int((x-bw/2)*W), int((y-bh/2)*H)),
                          (int((x+bw/2)*W), int((y+bh/2)*H)), COL[c], 3)
    tiles.append(cv2.resize(im, (400, 225)))
cv2_imshow(np.vstack([np.hstack(tiles[i:i+4]) for i in range(0, 12, 4)]))
print('초록 = fire, 파랑 = smoke')

In [ ]:
# [5] 학습 — YOLOv8s / 60 epoch / 640px (약 55분)
!yolo detect train model=yolov8s.pt data=ds6/data.yaml epochs=60 imgsz=640 \
    batch=16 project=/content/runs name=r6 exist_ok=False

In [ ]:
# [6] 채점 — 가장 최근 학습 결과를 자동으로 집는다
import glob, os
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
print('가중치:', best)
!python scripts/eval_gate6.py --weights "$best" --base assets/weights/round5_best.pt \
    --eval-dir eval --cctv assets/eval_neg --steam assets/eval_steam --conf 0.10

In [ ]:
# [7] 오탐·미탐 사례 보기 — 수증기 오탐이 이번 회차의 핵심 위험
import glob, os, cv2, numpy as np
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
m = YOLO(best); CONF = 0.10
NAME = {0: 'fire', 1: 'smoke'}; COL = {0: (0, 255, 0), 1: (255, 180, 0)}

def grid(paths, title, n=8, cols=4):
    hits = []
    for p in paths:
        r = m.predict(p, conf=CONF, verbose=False)[0]
        if len(r.boxes) == 0:
            continue
        im = cv2.imread(p)
        for b, c, k in zip(r.boxes.xyxy.cpu().numpy().astype(int),
                           r.boxes.conf.cpu().numpy(),
                           r.boxes.cls.cpu().numpy().astype(int)):
            cv2.rectangle(im, (b[0], b[1]), (b[2], b[3]), COL[k], 2)
            cv2.putText(im, f'{NAME[k]} {c:.2f}', (b[0], max(14, b[1]-4)),
                        cv2.FONT_HERSHEY_SIMPLEX, .5, COL[k], 2)
        hits.append(cv2.resize(im, (400, 225)))
        if len(hits) >= n:
            break
    print(f'\n■ {title} — {len(hits)}장 표시')
    if not hits:
        print('없음'); return
    while len(hits) % cols:
        hits.append(np.zeros((225, 400, 3), np.uint8))
    cv2_imshow(np.vstack([np.hstack(hits[i:i+cols]) for i in range(0, len(hits), cols)]))

grid(sorted(glob.glob('assets/eval_steam/*.jpg')), '수증기 오탐 (D)')
grid(sorted(glob.glob('assets/eval_neg/*.jpg')), '급식실 CCTV 오탐 (B)')
grid([l.strip() for l in open('eval/eval_smoke.txt')], '연기 정탐 (A-smoke)')

In [ ]:
# [8] 가중치 내려받기
import glob, os
from google.colab import files
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
files.download(best)